# Notebook 2 — Data Model & Test-Driven Development
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Understand the `data.py` module (`YFinanceAPI`, `SQLRepository`)
2. Set up a SQLite database to persist stock data
3. Practice TDD: write tests *before* (or alongside) the implementation
4. Insert and retrieve BSE data using `SQLRepository`
5. Verify the ETL pipeline end-to-end

## 1. Setup

In [ ]:
import sys
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Add src/ to path so we can import data.py and model.py
sys.path.insert(0, os.path.join('..', 'src'))

from data import YFinanceAPI, SQLRepository, get_stock_data

print('Imports OK')

# Paths
DB_PATH = os.path.join('..', 'database', 'stock_data.db')
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
print(f'Database path: {os.path.abspath(DB_PATH)}')

## 2. Understanding `data.py`

Our `src/data.py` provides three things:

| Class / Function | Responsibility |
|---|---|
| `YFinanceAPI` | Downloads OHLCV data from Yahoo Finance |
| `SQLRepository` | Reads/writes DataFrames to SQLite |
| `get_stock_data()` | One-call convenience wrapper |

Let's explore each.

In [ ]:
# ── YFinanceAPI ──────────────────────────────────────────────────────────────
api = YFinanceAPI()

# Fetch SENSEX data
df_sensex = api.get_daily_data('^BSESN', start='2020-01-01', end='2024-12-31')

print(f'Rows fetched : {len(df_sensex):,}')
print(f'Columns      : {list(df_sensex.columns)}')
print(f'Index type   : {type(df_sensex.index).__name__}')
print(f'Null values  : {df_sensex.isnull().sum().sum()}')
df_sensex.head()

## 3. SQLite Database Setup

We use SQLite (no server required) with `SQLRepository` to store our clean data.

In [ ]:
# Create SQLite connection and repository
connection = sqlite3.connect(DB_PATH)
repo = SQLRepository(connection=connection)

print(f'Connected to: {DB_PATH}')
print(f'SQLRepository instance: {repo}')

## 4. Insert Data into SQLite

In [ ]:
# Insert SENSEX — use 'replace' so we can re-run this cell safely
result = repo.insert_table(
    table_name='^BSESN',
    records=df_sensex,
    if_exists='replace'
)
print('Insert result:', result)

In [ ]:
# Insert a few more stocks — uncomment to run
STOCKS_TO_LOAD = [
    'RELIANCE.NS',
    'TCS.NS',
    'INFY.NS',
    'HDFCBANK.NS',
]

for ticker in STOCKS_TO_LOAD:
    data = api.get_daily_data(ticker, start='2015-01-01', end='2024-12-31')
    res = repo.insert_table(table_name=ticker, records=data, if_exists='replace')
    print(f'{ticker:<20}  →  {res["records_inserted"]:>4} rows inserted')

## 5. Read Data Back from SQLite

In [ ]:
# Read all SENSEX rows
df_from_db = repo.read_table('^BSESN')
print(f'Read {len(df_from_db):,} rows from DB')
print(f'Columns: {list(df_from_db.columns)}')
df_from_db.tail()

In [ ]:
# Read only the most recent 500 rows
df_recent = repo.read_table('^BSESN', limit=500)
print(f'Read {len(df_recent):,} rows (limited to 500)')
print(f'Date range: {df_recent.index.min().date()}  →  {df_recent.index.max().date()}')

## 6. Test-Driven Development (TDD)

TDD means: **write the test → run it (expect failure) → write code to make it pass → refactor**.

We write inline assertion-style tests here; the full pytest suite lives in `tests/`.

In [ ]:
# ── TDD helper ───────────────────────────────────────────────────────────────
def assert_equal(actual, expected, label=''):
    if actual == expected:
        print(f'  PASS  {label}')
    else:
        print(f'  FAIL  {label}  (got {actual!r}, expected {expected!r})')

def assert_true(condition, label=''):
    if condition:
        print(f'  PASS  {label}')
    else:
        print(f'  FAIL  {label}')

print('=== YFinanceAPI tests ===')

sample = api.get_daily_data('^BSESN', '2023-01-01', '2023-06-30')

assert_true(isinstance(sample, pd.DataFrame),
            'get_daily_data returns a DataFrame')
assert_true(isinstance(sample.index, pd.DatetimeIndex),
            'Index is DatetimeIndex')
assert_true('returns' in sample.columns,
            'returns column present')
assert_equal(sample['returns'].isna().sum(), 0,
             'No NaN in returns')
assert_true(len(sample) > 0,
            'Non-empty DataFrame returned')

In [ ]:
# ── TDD: SQLRepository ────────────────────────────────────────────────────────
print('=== SQLRepository tests ===')

# Use an in-memory DB so tests don't pollute the real DB
test_conn = sqlite3.connect(':memory:')
test_repo = SQLRepository(connection=test_conn)

test_df = pd.DataFrame(
    {'Open': [100.0, 101.0], 'Close': [101.0, 102.0], 'returns': [1.0, 0.99]},
    index=pd.to_datetime(['2023-01-02', '2023-01-03'])
)
test_df.index.name = 'Date'

insert_result = test_repo.insert_table('TEST', test_df, if_exists='replace')
assert_true(insert_result['transaction_successful'],
            'insert_table returns success')

read_back = test_repo.read_table('TEST')
assert_true(isinstance(read_back, pd.DataFrame),
            'read_table returns DataFrame')
assert_equal(len(read_back), 2,
             'Correct number of rows read back')

limited = test_repo.read_table('TEST', limit=1)
assert_equal(len(limited), 1,
             'limit parameter works')

assert_true(test_repo.table_exists('TEST'),
            'table_exists returns True for existing table')
assert_true(not test_repo.table_exists('NONEXISTENT'),
            'table_exists returns False for missing table')

test_conn.close()

## 7. Verify the Full ETL Pipeline

In [ ]:
# Full pipeline test: fetch → insert → read → compare
TICKER = 'WIPRO.NS'
START, END = '2022-01-01', '2024-01-01'

print(f'Step 1: Fetch {TICKER}...')
fetched = api.get_daily_data(TICKER, START, END)
print(f'  Fetched {len(fetched):,} rows')

print('Step 2: Insert into DB...')
result = repo.insert_table(TICKER, fetched, if_exists='replace')
print(f'  {result}')

print('Step 3: Read back from DB...')
loaded = repo.read_table(TICKER)
print(f'  Read {len(loaded):,} rows')

print('Step 4: Verify round-trip...')
close_match = (fetched['Close'].round(4) == loaded['Close'].round(4)).all()
print(f'  Close prices match: {close_match}')
print('\nETL pipeline is working correctly!')

## 8. Inspect the Database

In [ ]:
# List all tables in the database
tables = pd.read_sql(
    "SELECT name, type FROM sqlite_master WHERE type='table'",
    con=connection
)
print(f'Tables in {DB_PATH}:')
print(tables.to_string(index=False))

In [ ]:
# Row count per table
for tbl in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM '{tbl}'", con=connection).iloc[0]['n']
    print(f'  {tbl:<25}  {count:>5} rows')

In [ ]:
# Clean up connection
connection.close()
print('Connection closed.')

## Summary

- `YFinanceAPI.get_daily_data()` fetches and cleans BSE/NSE data reliably.
- `SQLRepository` persists and retrieves DataFrames to/from SQLite with a clean interface.
- All tests pass — the ETL layer is solid.
- Full pytest suite: `pytest tests/test_data.py -v`

**Next:** Notebook 3 — computing volatility and fitting the GARCH model.